In [1]:
#!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

## 0. Preparation

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv("course_lead_scoring_2026.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [41]:
df.info()
df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employment_status         4806 non-null   str    
 3   location                  4792 non-null   str    
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB


lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [3]:
na_values = {"lead_score": 0.0, "annual_income": 0.0, "lead_source": "NA", 
             "industry": "NA", "employment_status": "NA", "location": "NA"}
df = df.fillna(value=na_values)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               5000 non-null   str    
 1   industry                  5000 non-null   str    
 2   employment_status         5000 non-null   str    
 3   location                  5000 non-null   str    
 4   annual_income             5000 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                5000 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB


## 1. The most frequent observation

In [4]:
df['industry'].value_counts() 

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

## 2. Correlation

In [5]:
numerical = ["annual_income", "number_of_courses_viewed", "interaction_count",
            "lead_score", "converted"]
categorical = ["lead_source", "industry", "employment_status", "location"]

corr_matrix = df[numerical].corr()
print(corr_matrix)

                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.161300   
number_of_courses_viewed       0.161300                  1.000000   
interaction_count              0.122842                  0.721609   
lead_score                     0.229496                  0.757204   
converted                      0.189652                  0.395038   

                          interaction_count  lead_score  converted  
annual_income                      0.122842    0.229496   0.189652  
number_of_courses_viewed           0.721609    0.757204   0.395038  
interaction_count                  1.000000    0.915746   0.448624  
lead_score                         0.915746    1.000000   0.483246  
converted                          0.448624    0.483246   1.000000  


### Split the data

In [6]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

X_train = df_train.drop('converted', axis=1)
X_test = df_test.drop('converted', axis=1)
X_val = df_val.drop('converted', axis=1)

y_train = df_train["converted"]
y_test = df_test["converted"]
y_val = df_val["converted"]

## 3. Mutual info score

In [7]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

mi = df_train[categorical].apply(mutual_info_converted_score)
mi = mi.sort_values(ascending=False).round(2)
mi

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

## 4. Training

### One-hot encoding

In [8]:
dicts = df[categorical + numerical].to_dict(orient='records')

dv = DictVectorizer(sparse=False)

numerical_X = ["annual_income", "number_of_courses_viewed", "interaction_count",
            "lead_score"]
train_dict = X_train[categorical + numerical_X].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = X_val[categorical + numerical_X].to_dict(orient='records')
X_val = dv.transform(val_dict)

test_dict = X_test[categorical + numerical_X].to_dict(orient='records')
X_test = dv.transform(test_dict)

In [22]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
is_converted = (y_pred >= 0.5)

accuracy = (y_val.values == is_converted).mean()
round(accuracy, 2)

np.float64(0.64)

## 5. Feature elimination

In [23]:
feature_names = dv.get_feature_names_out()
feature_names

accuracy

np.float64(0.645)

In [24]:
feature_names = dv.get_feature_names_out()

X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_val_df = pd.DataFrame(X_val, columns=feature_names)

features_to_exclude = ['number_of_courses_viewed', 'interaction_count']

for feature in features_to_exclude:
    new_X_train = X_train_df.drop(feature, axis=1)
    new_X_val = X_val_df.drop(feature, axis=1) 
    
    new_model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    new_model.fit(new_X_train, y_train)

    y_pred = new_model.predict_proba(new_X_val)[:, 1]
    is_converted = (y_pred >= 0.5)

    new_accuracy = (y_val.values == is_converted).mean()
    print(f'feature excluded: {feature}, new_accuracy: {new_accuracy}, difference: {round(new_accuracy, 2) - round(accuracy, 2)}')

feature excluded: number_of_courses_viewed, new_accuracy: 0.643, difference: 0.0
feature excluded: interaction_count, new_accuracy: 0.601, difference: -0.040000000000000036


## 6. Evaluation on test

In [25]:
c_values = [0.000001, 0.00001, 0.0001, 0.001]

for c in c_values:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict_proba(X_val)[:, 1]
    is_converted = (y_pred >= 0.5)
    
    accuracy = (y_val.values == is_converted).mean()
    rounded_accuracy = round(accuracy, 3)
    
    print(f'C={c}: accuracy = {rounded_accuracy}')

C=1e-06: accuracy = 0.598
C=1e-05: accuracy = 0.598
C=0.0001: accuracy = 0.613
C=0.001: accuracy = 0.645
